# Data Cleaning



In [2]:
import pandas as pd
import os
import glob
import re

CLEAN = 'clean'
os.makedirs(CLEAN, exist_ok=True)

## Helper Functions

In [3]:
def load_csvs(directory, pattern="*.csv", id_dtype=None):
    """Load all CSVs from a directory into one dataframe."""
    files = glob.glob(os.path.join(directory, pattern))
    dfs = []
    for f in sorted(files):
        try:
            dtype = {id_dtype: str} if id_dtype else None
            dfs.append(pd.read_csv(f, encoding="utf-8", dtype=dtype))
        except Exception as e:
            print(f"  skip {f}: {e}")
    if not dfs:
        print(f"  no files found in {directory}/{pattern}")
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def clean_text(series):
    """Text cleaning: strip HTML, normalise whitespace, drop empty."""
    s = series.fillna('').astype(str)
    s = s.apply(lambda x: re.sub(r'<[^>]+>', '', x))
    s = s.apply(lambda x: re.sub(r'\s+', ' ', x).strip())
    s = s.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    return s


def summarise(df, name, id_col, author_col, date_col='created_at'):
    """Print a summary of a cleaned dataframe."""
    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"  Rows:           {len(df):,}")
    if id_col in df.columns:
        print(f"  Unique IDs:     {df[id_col].nunique():,}")
    if author_col and author_col in df.columns:
        print(f"  Unique authors: {df[author_col].nunique():,}")
    if date_col and date_col in df.columns:
        print(f"  Date range:     {df[date_col].min()} to {df[date_col].max()}")
    print(f"  Nulls per col:")
    nulls = df.isnull().sum()
    for col in nulls[nulls > 0].index:
        print(f"    {col}: {nulls[col]:,}")

##  Bluesky Posts

In [4]:
# LOAD
bsky = load_csvs("bluesky_posts")
print(f"Loaded: {len(bsky):,} rows")

# DEDUPLICATE
before = len(bsky)
bsky = bsky.drop_duplicates(subset=['post_uri'], keep='first')
print(f"Deduped: {before:,} -> {len(bsky):,}  (removed {before - len(bsky):,})")

# PARSE DATES
bsky['created_at'] = pd.to_datetime(bsky['created_at'], format='mixed', utc=True)

# CLEAN TEXT
bsky['text'] = clean_text(bsky['text'])

# DROP EMPTY TEXT
before = len(bsky)
bsky = bsky.dropna(subset=['text'])
print(f"Dropped empty text: {before:,} -> {len(bsky):,}")

# FILTER ENGLISH ONLY (where language tag exists)
if 'language' in bsky.columns:
    n_with_lang = bsky['language'].notna().sum()
    n_en = bsky[bsky['language'] == 'en'].shape[0]
    print(f"\nLanguage tags: {n_with_lang:,} posts have one")
    print(f"English posts: {n_en:,}")
    # keep english + posts with no language tag (rather than losing them)
    bsky = bsky[(bsky['language'] == 'en') | (bsky['language'].isna())]
    print(f"After English filter: {len(bsky):,}")

# STANDARDISE KEYWORD COLUMN
bsky['search_keyword'] = bsky['search_keyword'].str.strip()

# ADD PLATFORM LABEL
bsky['platform'] = 'bluesky'

summarise(bsky, 'BLUESKY POSTS (CLEAN)', 'post_uri', 'author_did')
print(f"\nPosts per keyword:")
print(bsky['search_keyword'].value_counts().to_string())

Loaded: 2,112,061 rows
Deduped: 2,112,061 -> 2,035,552  (removed 76,509)
Dropped empty text: 2,035,552 -> 2,031,607

Language tags: 1,655,825 posts have one
English posts: 1,153,322
After English filter: 1,529,104

BLUESKY POSTS (CLEAN)
  Rows:           1,529,104
  Unique IDs:     1,529,104
  Unique authors: 341,317
  Date range:     2023-02-01 00:26:47+00:00 to 2026-05-29 01:31:25+00:00
  Nulls per col:
    language: 375,782
    author_display_name: 91,707
    author_avatar: 9,569
    author_description: 1,529,104

Posts per keyword:
search_keyword
AI                        514432
ChatGPT                   373945
GenAI                     138090
Midjourney                101630
AI slop                   100541
AI powered                 72784
AI agents                  58741
anti-AI                    45869
pro-AI                     44333
Deepfake                   38069
BigTech                    11773
AI data center              6990
AI Ethics                   6371
Claude AI     

In [5]:
# SAVE
bsky.to_csv('clean/bsky_posts_clean.csv', index=False, encoding='utf-8')
print(f"Saved clean/bsky_posts_clean.csv ({len(bsky):,} rows)")

Saved clean/bsky_posts_clean.csv (1,529,104 rows)


##  Truth Social Posts

In [6]:
# LOAD
truth = load_csvs("truth_posts", pattern="truth_*.csv", id_dtype="post_id")
print(f"Loaded: {len(truth):,} rows")

# FIX CORRUPTED IDS (scientific notation from Excel)
if 'post_id' in truth.columns and 'post_url' in truth.columns:
    corrupted = truth['post_id'].str.contains('e\+', na=False)
    if corrupted.any():
        extracted = truth.loc[corrupted, 'post_url'].str.extract(r'/(\d+)$')[0]
        truth.loc[corrupted, 'post_id'] = extracted
        print(f"Fixed {corrupted.sum():,} corrupted post IDs")

# DEDUPLICATE
before = len(truth)
truth = truth.drop_duplicates(subset=['post_id'], keep='first')
print(f"Deduped: {before:,} -> {len(truth):,}  (removed {before - len(truth):,})")

# PARSE DATES
truth['created_at'] = pd.to_datetime(truth['created_at'], format='mixed', utc=True)

# CLEAN TEXT
truth['text'] = clean_text(truth['text'])

# DROP EMPTY TEXT
before = len(truth)
truth = truth.dropna(subset=['text'])
print(f"Dropped empty text: {before:,} -> {len(truth):,}")

# FILTER ENGLISH ONLY
if 'language' in truth.columns:
    n_with_lang = truth['language'].notna().sum()
    n_en = truth[truth['language'] == 'en'].shape[0]
    print(f"\nLanguage tags: {n_with_lang:,} posts have one")
    print(f"English posts: {n_en:,}")
    truth = truth[(truth['language'] == 'en') | (truth['language'].isna())]
    print(f"After English filter: {len(truth):,}")

# STANDARDISE HASHTAG COLUMN
truth['hashtag'] = truth['hashtag'].str.strip()

# ADD PLATFORM LABEL
truth['platform'] = 'truth_social'

summarise(truth, 'TRUTH SOCIAL POSTS (CLEAN)', 'post_id', 'author_id')
print(f"\nPosts per hashtag:")
print(truth['hashtag'].value_counts().to_string())

<>:7: SyntaxWarning: invalid escape sequence '\+'
<>:7: SyntaxWarning: invalid escape sequence '\+'
/var/folders/5d/hqrlt26d7nj7ktpwgl8gy1900000gn/T/ipykernel_2270/3374926120.py:7: SyntaxWarning: invalid escape sequence '\+'
  corrupted = truth['post_id'].str.contains('e\+', na=False)


Loaded: 55,108 rows
Deduped: 55,108 -> 50,875  (removed 4,233)
Dropped empty text: 50,875 -> 50,875

Language tags: 50,779 posts have one
English posts: 47,990
After English filter: 48,086

TRUTH SOCIAL POSTS (CLEAN)
  Rows:           48,086
  Unique IDs:     48,086
  Unique authors: 6,276
  Date range:     2022-02-15 17:04:41.144000+00:00 to 2026-05-19 16:08:33.224000+00:00
  Nulls per col:
    language: 96
    author_display_name: 452

Posts per hashtag:
hashtag
AI                        23560
BigTech                   12687
ArtificialIntelligence     6395
ChatGPT                    2797
project2025                 969
deepfake                    712
Midjourney                  370
AIagents                    178
AIethics                    167
AIdatacenter                 88
GenAI                        47
AIpowered                    44
ClaudeAI                     43
AIslop                       17
AIenergy                      5
AIbreakthrough                2
AIhallucination    

In [7]:
# SAVE
truth.to_csv('clean/truth_posts_clean.csv', index=False, encoding='utf-8')
print(f"Saved clean/truth_posts_clean.csv ({len(truth):,} rows)")

Saved clean/truth_posts_clean.csv (48,086 rows)


##  Bluesky Comments

In [8]:
# LOAD
bsky_comments = load_csvs("bluesky_comments")
print(f"Loaded: {len(bsky_comments):,} rows")

if not bsky_comments.empty:
    # DEDUPLICATE
    before = len(bsky_comments)
    bsky_comments = bsky_comments.drop_duplicates(subset=['comment_uri'], keep='first')
    print(f"Deduped: {before:,} -> {len(bsky_comments):,}")

    # PARSE DATES
    bsky_comments['created_at'] = pd.to_datetime(bsky_comments['created_at'], format='mixed', utc=True)

    # CLEAN TEXT
    bsky_comments['text'] = clean_text(bsky_comments['text'])
    before = len(bsky_comments)
    bsky_comments = bsky_comments.dropna(subset=['text'])
    print(f"Dropped empty text: {before:,} -> {len(bsky_comments):,}")

    # FILTER ENGLISH
    if 'language' in bsky_comments.columns:
        bsky_comments = bsky_comments[(bsky_comments['language'] == 'en') | (bsky_comments['language'].isna())]
        print(f"After English filter: {len(bsky_comments):,}")

    bsky_comments['platform'] = 'bluesky'

    summarise(bsky_comments, 'BLUESKY COMMENTS (CLEAN)', 'comment_uri', 'author_did')
    print(f"Max thread depth: {bsky_comments['depth'].max()}")

    bsky_comments.to_csv('clean/bsky_comments_clean.csv', index=False, encoding='utf-8')
    print(f"\nSaved clean/bsky_comments_clean.csv ({len(bsky_comments):,} rows)")

  skip bluesky_comments/comments_batch_20260605_163340.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_163724.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_163916.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_164300.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_164452.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_164643.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_164835.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_165027.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_165219.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_165411.csv: [Errno 60] Operation timed out
  skip bluesky_comments/comments_batch_20260605_165603.csv: [Errno 60] Operation timed out

##  Truth Social Comments

In [9]:
# LOAD
truth_comments = load_csvs("truth_comments")
print(f"Loaded: {len(truth_comments):,} rows")

if not truth_comments.empty:
    # DEDUPLICATE
    before = len(truth_comments)
    truth_comments = truth_comments.drop_duplicates(subset=['comment_id'], keep='first')
    print(f"Deduped: {before:,} -> {len(truth_comments):,}")

    # PARSE DATES
    if 'created_at' in truth_comments.columns:
        truth_comments['created_at'] = pd.to_datetime(truth_comments['created_at'], format='mixed', utc=True)

    # CLEAN TEXT
    truth_comments['text'] = clean_text(truth_comments['text'])
    before = len(truth_comments)
    truth_comments = truth_comments.dropna(subset=['text'])
    print(f"Dropped empty text: {before:,} -> {len(truth_comments):,}")

    truth_comments['platform'] = 'truth_social'

    summarise(truth_comments, 'TRUTH SOCIAL COMMENTS (CLEAN)', 'comment_id', 'author_handle')

    truth_comments.to_csv('clean/truth_comments_clean.csv', index=False, encoding='utf-8')
    print(f"\nSaved clean/truth_comments_clean.csv ({len(truth_comments):,} rows)")

Loaded: 4,403 rows
Deduped: 4,403 -> 4,403
Dropped empty text: 4,403 -> 4,403

TRUTH SOCIAL COMMENTS (CLEAN)
  Rows:           4,403
  Unique IDs:     4,403
  Unique authors: 3,351
  Date range:     NaT to NaT
  Nulls per col:
    created_at: 4,403

Saved clean/truth_comments_clean.csv (4,403 rows)


## Bluesky Reposts

In [10]:
# LOAD
bsky_reposts = load_csvs("bluesky_reposts")
print(f"Loaded: {len(bsky_reposts):,} rows")

if not bsky_reposts.empty:
    # DEDUPLICATE (same user reposting same post)
    before = len(bsky_reposts)
    bsky_reposts = bsky_reposts.drop_duplicates(subset=['post_uri', 'reposted_by_did'], keep='first')
    print(f"Deduped: {before:,} -> {len(bsky_reposts):,}")

    bsky_reposts['platform'] = 'bluesky'

    print(f"\n{'=' * 50}")
    print("BLUESKY REPOSTS (CLEAN)")
    print(f"  Rows:           {len(bsky_reposts):,}")
    print(f"  Unique reposters: {bsky_reposts['reposted_by_did'].nunique():,}")
    print(f"  Posts covered:    {bsky_reposts['post_uri'].nunique():,}")

    bsky_reposts.to_csv('clean/bsky_reposts_clean.csv', index=False, encoding='utf-8')
    print(f"\nSaved clean/bsky_reposts_clean.csv ({len(bsky_reposts):,} rows)")

  skip bluesky_reposts/reposts_batch_20260605_212705.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_213007.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_213305.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_213601.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_213857.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_214156.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_214455.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_214749.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_215057.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_215407.csv: [Errno 60] Operation timed out
  skip bluesky_reposts/reposts_batch_20260605_215705.csv: [Errno 60] Operation timed out
  skip bluesky_repost

## Truth Social Reposts

In [11]:
# LOAD
truth_reposts = load_csvs("truth_reposts")
print(f"Loaded: {len(truth_reposts):,} rows")

if not truth_reposts.empty:
    # DEDUPLICATE
    before = len(truth_reposts)
    truth_reposts = truth_reposts.drop_duplicates(subset=['post_id', 'retruthed_by_id'], keep='first')
    print(f"Deduped: {before:,} -> {len(truth_reposts):,}")

    truth_reposts['platform'] = 'truth_social'

    print(f"\n{'=' * 50}")
    print("TRUTH SOCIAL REPOSTS (CLEAN)")
    print(f"  Rows:           {len(truth_reposts):,}")
    print(f"  Unique reposters: {truth_reposts['retruthed_by_id'].nunique():,}")
    print(f"  Posts covered:    {truth_reposts['post_id'].nunique():,}")

    truth_reposts.to_csv('clean/truth_reposts_clean.csv', index=False, encoding='utf-8')
    print(f"\nSaved clean/truth_reposts_clean.csv ({len(truth_reposts):,} rows)")

Loaded: 34,266 rows
Deduped: 34,266 -> 34,266

TRUTH SOCIAL REPOSTS (CLEAN)
  Rows:           34,266
  Unique reposters: 15,722
  Posts covered:    5,082

Saved clean/truth_reposts_clean.csv (34,266 rows)


## Final Summary


In [12]:
print("CLEAN DATA SUMMARY")
print("=" * 50)

for fname in sorted(glob.glob('clean/*.csv')):
    df = pd.read_csv(fname, nrows=0)
    nrows = sum(1 for _ in open(fname)) - 1
    size_mb = os.path.getsize(fname) / (1024 * 1024)
    print(f"\n{os.path.basename(fname)}")
    print(f"  Rows:    {nrows:,}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Size:    {size_mb:.1f} MB")
    print(f"  Fields:  {', '.join(df.columns[:8])}{'...' if len(df.columns) > 8 else ''}")

CLEAN DATA SUMMARY

bsky_aligned.csv
  Rows:    75,996
  Columns: 19
  Size:    42.9 MB
  Fields:  post_uri, post_cid, text, created_at, reply_count, repost_count, like_count, quote_count...

bsky_aligned_top.csv
  Rows:    3,803
  Columns: 29
  Size:    2.5 MB
  Fields:  post_uri, post_cid, text, created_at, reply_count, repost_count, like_count, quote_count...

bsky_comments_clean.csv
  Rows:    438,005
  Columns: 23
  Size:    277.4 MB
  Fields:  root_post_uri, parent_uri, comment_uri, comment_cid, depth, text, created_at, language...

bsky_posts_clean.csv
  Rows:    1,529,132
  Columns: 19
  Size:    861.7 MB
  Fields:  post_uri, post_cid, text, created_at, reply_count, repost_count, like_count, quote_count...

bsky_repost_edgelist.csv
  Rows:    1,577,684
  Columns: 3
  Size:    102.3 MB
  Fields:  source, target, weight

bsky_repost_edgelist_updated.csv
  Rows:    1,577,684
  Columns: 9
  Size:    117.5 MB
  Fields:  source, target, weight, source_bias, source_factuality, source_

## Check samples


In [13]:
for fname in sorted(glob.glob('clean/*.csv')):

    print(f"{os.path.basename(fname)}")
    print("=" * 50)
    df = pd.read_csv(fname, nrows=5)
    if 'text' in df.columns:
        for i, row in df.iterrows():
            print(f"\n[{i}] {row.get('created_at', 'no date')}")
            print(f"    {str(row['text'])[:150]}")
    else:
        print(df.head())

bsky_aligned.csv

[0] 2023-03-14 00:04:54.685000+00:00
    It’s amazing how, one way or another, letting myself log into Twitter always fills me with anxiety and makes me less able to enjoy life. Yes, there’s 

[1] 2023-03-15 02:49:45.020000+00:00
    feeling the need for a candlelit dinner after a week of digital bank runs and advances in AI. i can only tolerate modernity by keeping one foot firmly

[2] 2023-03-27 12:48:56.706000+00:00
    TL;DR Next Evolution Source: https://marketoonist.com/2023/03/ai-written-ai-read.html

[3] 2023-03-24 14:30:40.715000+00:00
    It’s a weird vibe to write code your proud of knowing that AI coding is basically here

[4] 2023-03-30 08:45:44.738000+00:00
    Bird app lately is so many bad takes about AI.
bsky_aligned_top.csv

[0] 2023-04-23 13:31:15.728000+00:00
    É como diz o ditado: SEM ANISTIA! Vem aí a CPMI dos Golpistas. Arte perfeita do @crisvector.bsky.social

[1] 2023-05-01 13:30:30.542000+00:00
    A guy that is about to show you some of th